### Setup

In [1]:
question_type = "open-ended" # Options: "open-ended"
dataset = "ptpt" # Options: "ptpt", "ptbr"
prompt_language = dataset # Options: "ptpt", "ptbr", "en"
# models = {"claude-haiku-4.5", "deepseek-chat-v3.1", "gemini-2.5-flash", "gemma-3-27b-it", "gpt-5","llama-3.3-70b-instruct", "qwen3-8b", "qwen3-14b", "qwen3-32b", "qwen3-30b-a3b", "qwen3-235b-a22b"}
models = {"qwen3-14b"}
judge = "qwen3-14b"

### Load Judge Answers

In [2]:
import json

# Build judge_responses with parsed judge scores/explanations
judge_responses = {model: [] for model in models}

for model in models:
    judge_responses_file = f'{model}-{dataset}-judge-responses-prompt-language-{prompt_language}-judge-{judge}.json'
    with open(f'judge-results/{judge_responses_file}', 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():  # skip empty lines
                judge_responses[model].append(json.loads(line))

### Load Judge Prompts

In [3]:
judge_prompts = []

prompts_file = f'{model}-{dataset}-judge-prompts-prompt-language-{prompt_language}.json'
with open(f'judge-prompts/{prompts_file}', 'r', encoding='utf-8') as f:
    judge_prompts = json.load(f)

### Parse Judge Answers

In [ ]:
import json
import re

def safe_load_json_with_backslashes(text, debug=False):
    # 1) Try raw
    try:
        return json.loads(text)
    except json.JSONDecodeError as e1:
        if debug:
            print("raw json.loads failed:", e1)

    # 2) Try targeted regex: escape any backslash not followed by a valid JSON escape
    try:
        fixed = re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', text)
        return json.loads(fixed)
    except json.JSONDecodeError as e2:
        if debug:
            print("regex fix failed:", e2)
            # show a short preview to help debugging
            print("preview (repr, first 300 chars):", repr(text)[:300])

    # 3) Aggressive fallback: double every backslash
    try:
        very_fixed = text.replace('\\', '\\\\')
        return json.loads(very_fixed)
    except json.JSONDecodeError as e3:
        # 4) Give diagnostic info and re-raise so caller can inspect
        if debug:
            print("full replace fix failed:", e3)
            print("full repr(text):")
            print(repr(text))
        raise  # caller can catch and decide what to do

error_flag = False

# usage in your loop
for model in models:
    for response in judge_responses[model]:
        try:
            text = response["raw_response"]["choice.message.content"]
            try:
                data = safe_load_json_with_backslashes(text, debug=False)
            except json.JSONDecodeError:
                # last resort: print repr for this item so you can inspect exactly why it still fails
                print("FINAL FAIL: repr(text) for id", response.get('id'))
                print(repr(text))
                response['score'] = None
                response['explanation'] = None
                continue

            # now populate
            response['score'] = data.get("score")
            response['explanation'] = data.get("explanation")
        except KeyError:
            print(f"Question didn't reach model: {response.get('id')}")
            error_flag = True

if error_flag:
    raise RuntimeError("Some questions did not reach the model. Please check the logs above.")


### Calculate Accuracies

In [ ]:
from collections import defaultdict

# Build golden lookup by question id
judge_prompts_by_id = {item["id"]: item for item in judge_prompts}


# Helper to initialize stats dict
def init_stats():
    return {model: {lvl: {"correct": 0, "total": 0} for lvl in range(1, 5)} for model in models}

# We keep figure/no-figure splits plus overall
stats_with_fig = init_stats()
stats_no_fig = init_stats()
stats_all = init_stats()  # global (figure + no figure)

for model in models:
    for r in judge_responses.get(model, []):
        qid = r.get("id")
        p = judge_prompts_by_id.get(qid)
        if p is None:
            continue
        # Level handling (ensure integer 1..4)
        lvl_raw = p.get("level", 0)
        try:
            lvl = int(lvl_raw)
        except Exception:
            continue
        if lvl not in (1, 2, 3, 4):
            continue

        # Figure presence (any of the possible fields)
        contains_figure = bool(
            p.get("contains_latex_figure_in_question")
        )

        # Always update global stats
        stats_all[model][lvl]["total"] += 1
        stats_all[model][lvl]["correct"] += r.get("score")

        # Split by figure presence
        if contains_figure:
            stats_with_fig[model][lvl]["total"] += 1
            stats_with_fig[model][lvl]["correct"] += r.get("score")
        else:
            stats_no_fig[model][lvl]["total"] += 1
            stats_no_fig[model][lvl]["correct"] += r.get("score")



def compute_acc(stats):
    out = {}
    for model in models:
        out[model] = {}
        for lvl in range(1, 5):
            c = stats[model][lvl]["correct"]
            t = stats[model][lvl]["total"]
            out[model][lvl] = {
                "correct": c,
                "total": t,
                "accuracy": (c / t * 100.0) if t else None,
            }
    return out


acc_with_fig = compute_acc(stats_with_fig)
acc_no_fig = compute_acc(stats_no_fig)
acc_all = compute_acc(stats_all)  # global


def format_acc(title, acc):
    lines = [f"=== {title} ==="]
    for model in sorted(models):
        lines.append("")
        lines.append(f"Model: {model}")
        for lvl in range(1, 5):
            a = acc[model][lvl]
            t = a["total"]
            if not t:
                lines.append(f"  Level {lvl}: no data")
            else:
                lines.append(
                    f"  Level {lvl}: {a['correct']}/{t} correct ({a['accuracy']:.2f}%)"
                )
    lines.append("")
    return "\n".join(lines)


report_text = []
report_text.append(format_acc("Overall (fig and no fig)", acc_all))
report_text.append(format_acc("With figures only", acc_with_fig))
report_text.append(format_acc("Without figures", acc_no_fig))
report_text = "\n".join(report_text)

### Save accuracy results

In [ ]:
# Save to results directory with requested filename pattern
out_filename = f"{dataset}-{question_type}-all-models-prompt-language-{prompt_language}-acc-report.txt"
out_path = f"results/accuracy-reports/{out_filename}"
with open(out_path, "w", encoding="utf-8") as out:
    out.write(report_text)